# 02 - Stage 6.2: Attention U-Net evaluation

Uses the trained checkpoint and saved val/calibration results. If the results or 84 predicted masks are missing, it calls `src.predict_seg` on the available CUDA device (or CPU fallback). It does not retrain the model.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from IPython.display import Image, display

root = Path.cwd()
split = root / "results" / "review1_local" / "full281" / "split_281cases.json"
out = root / "results" / "final"
mask_dir = out / "masks"
mask_count = len(list(mask_dir.glob("pancreas_*.npy")))
if mask_count < 84 or not (out / "seg_val" / "seg_summary.json").exists() or not (out / "seg_calib" / "seg_summary.json").exists():
    for group in ("val", "calib"):
        subprocess.run([
            sys.executable, "-m", "src.predict_seg", "--split", str(split),
            "--group", group, "--save-masks", str(mask_dir),
            "--out", str(out / ("seg_" + group)),
        ], check=True, cwd=root)

for group in ("val", "calib"):
    summary = json.loads((out / ("seg_" + group) / "seg_summary.json").read_text(encoding="utf-8"))
    print(group, json.dumps(summary.get("by_size_bucket", summary), indent=2))
display(Image(filename=str(out / "fig_stage62_training.png")))
display(Image(filename=str(out / "seg_val" / "pred_pancreas_040.png")))